## Config

In [1]:
import pandas as pd
import numpy as np
import re
import time
import os
from dotenv import load_dotenv


from huggingface_hub import login

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [2]:
raw_path = os.path.join("data", "raw", 'projetos-lattes.csv')
dt = pd.read_csv(raw_path, sep=';', encoding='utf-8-sig')
dt.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
0,4430958315746203,Ivan do Carmo Machado,2024,Atual,Uma Investigação Empírica da Relação entre Ref...,Descrição: A temática de refatoração tem sido ...
1,4430958315746203,Ivan do Carmo Machado,2024,Atual,Consolidação da Rede Internacional de Pesquisa...,Descrição: O objetivo do projeto é promover a ...
2,4430958315746203,Ivan do Carmo Machado,2023,Atual,Melhoria da Qualidade de Software na Perspecti...,Descrição: O projeto vista investigar test sme...
3,4430958315746203,Ivan do Carmo Machado,2023,Atual,"SI2 - Instituto de Ciência, Tecnologia e Inova...",Descrição: Este projeto tem como objetivo gera...
4,4430958315746203,Ivan do Carmo Machado,2023,Atual,ALICE: Uma infrAestrutura para meLhoria da qua...,Descrição: Este projeto propõe a construção de...


In [3]:
dt.info()

<class 'pandas.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   lattes_id      171 non-null    int64
 1   professor      171 non-null    str  
 2   ano_inicio     171 non-null    int64
 3   ano_conclusao  171 non-null    str  
 4   projeto        171 non-null    str  
 5   descricao      171 non-null    str  
dtypes: int64(2), str(4)
memory usage: 8.1 KB


In [4]:
dt.shape

(171, 6)

In [5]:
dt.isna().sum()

lattes_id        0
professor        0
ano_inicio       0
ano_conclusao    0
projeto          0
descricao        0
dtype: int64

# Raw Data Extraction

### Extraction hard coded

lattes_id: ID Lattes do professor.

professor: Nome do docente.

titulo_projeto: Nome completo do projeto.

ano_inicio: Ano de início.

ano_conclusao: Ano de término (ou "Atual").

situacao: Em andamento / Concluído.

funcao: Coordenador / Integrante.

financiadores_lattes: Lista de agências/empresas financiadoras declaradas.

natureza: Pesquisa / Desenvolvimento / Extensão.

descricao_completa: Texto descritivo bruto do Lattes.

In [6]:
def extrair_metadados(texto):
    if not isinstance(texto, str):
        return pd.Series([None]*6)
    
    situacao = re.search(r'Situação:\s*([^;.]+)', texto)
    natureza = re.search(r'Natureza:\s*([^;.]+)', texto)
    alunos = re.search(r'Alunos envolvidos:\s*(.*?)(?=\.\s*Integrantes:|\.\s*Financiador|\Z)', texto)
    integrantes = re.search(r'Integrantes:\s*(.*?)(?=\.\s*Financiador|\Z)', texto)
    financiadores = re.search(r'Financiador\(es\):\s*(.+)', texto)
    desc_pura = re.search(r'Descrição:\s*(.*?)(?=\.\s*Situação:|\Z)', texto)

    return pd.Series([
        situacao.group(1).strip() if situacao else None,
        natureza.group(1).strip() if natureza else None,
        alunos.group(1).strip() if alunos else None,
        integrantes.group(1).strip() if integrantes else None,
        financiadores.group(1).strip() if financiadores else None,
        desc_pura.group(1).strip() if desc_pura else texto
    ])

In [7]:
df = dt.copy(deep=True)
df[['situacao', 'natureza', 'alunos_envolvidos', 'integrantes', 'financiadores', 'descricao_limpa']] = df['descricao'].apply(extrair_metadados)
df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa
0,4430958315746203,Ivan do Carmo Machado,2024,Atual,Uma Investigação Empírica da Relação entre Ref...,Descrição: A temática de refatoração tem sido ...,Em andamento,Pesquisa,Mestrado acadêmico: (3) Doutorado: (4),Ivan do Carmo Machado - Coordenador / Carla Il...,Conselho Nacional de Desenvolvimento Científic...,A temática de refatoração tem sido amplamente ...
1,4430958315746203,Ivan do Carmo Machado,2024,Atual,Consolidação da Rede Internacional de Pesquisa...,Descrição: O objetivo do projeto é promover a ...,Em andamento,Pesquisa,Mestrado acadêmico: (26) Doutorado: (26),Ivan do Carmo Machado - Integrante / Climene L...,NaN,O objetivo do projeto é promover a formação ac...
2,4430958315746203,Ivan do Carmo Machado,2023,Atual,Melhoria da Qualidade de Software na Perspecti...,Descrição: O projeto vista investigar test sme...,Em andamento,Pesquisa,Graduação: (2) / Mestrado acadêmico: (2) / Dou...,Ivan do Carmo Machado - Integrante / rohit Ghe...,Conselho Nacional de Desenvolvimento Científic...,O projeto vista investigar test smells no cont...
3,4430958315746203,Ivan do Carmo Machado,2023,Atual,"SI2 - Instituto de Ciência, Tecnologia e Inova...",Descrição: Este projeto tem como objetivo gera...,Em andamento,Pesquisa,NaN,Ivan do Carmo Machado - Integrante / Eduardo S...,NaN,Este projeto tem como objetivo geral a promoçã...
4,4430958315746203,Ivan do Carmo Machado,2023,Atual,ALICE: Uma infrAestrutura para meLhoria da qua...,Descrição: Este projeto propõe a construção de...,Em andamento,Pesquisa,Graduação: (2) / Doutorado: (3),Ivan do Carmo Machado - Coordenador / Larissa ...,Conselho Nacional de Desenvolvimento Científic...,Este projeto propõe a construção de uma infrae...


In [8]:
df["ano_conclusao"].unique()

<StringArray>
['Atual',  '2024',  '2022',  '2014',  '2013',  '2012',  '2025',  '2020',
  '2018',  '2004',  '2021',  '2016',  '2015',  '2010',  '2008',  '2006',
  '2019',  '2002',  '2017',  '2023',  '2026']
Length: 21, dtype: str

In [9]:
df["ano_conclusao"] = np.where(df["ano_conclusao"] == "Atual", np.nan, df["ano_conclusao"])
df["ano_conclusao"] = pd.to_numeric(df["ano_conclusao"])
df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa
0,4430958315746203,Ivan do Carmo Machado,2024,NaN,Uma Investigação Empírica da Relação entre Ref...,Descrição: A temática de refatoração tem sido ...,Em andamento,Pesquisa,Mestrado acadêmico: (3) Doutorado: (4),Ivan do Carmo Machado - Coordenador / Carla Il...,Conselho Nacional de Desenvolvimento Científic...,A temática de refatoração tem sido amplamente ...
1,4430958315746203,Ivan do Carmo Machado,2024,NaN,Consolidação da Rede Internacional de Pesquisa...,Descrição: O objetivo do projeto é promover a ...,Em andamento,Pesquisa,Mestrado acadêmico: (26) Doutorado: (26),Ivan do Carmo Machado - Integrante / Climene L...,NaN,O objetivo do projeto é promover a formação ac...
2,4430958315746203,Ivan do Carmo Machado,2023,NaN,Melhoria da Qualidade de Software na Perspecti...,Descrição: O projeto vista investigar test sme...,Em andamento,Pesquisa,Graduação: (2) / Mestrado acadêmico: (2) / Dou...,Ivan do Carmo Machado - Integrante / rohit Ghe...,Conselho Nacional de Desenvolvimento Científic...,O projeto vista investigar test smells no cont...
3,4430958315746203,Ivan do Carmo Machado,2023,NaN,"SI2 - Instituto de Ciência, Tecnologia e Inova...",Descrição: Este projeto tem como objetivo gera...,Em andamento,Pesquisa,NaN,Ivan do Carmo Machado - Integrante / Eduardo S...,NaN,Este projeto tem como objetivo geral a promoçã...
4,4430958315746203,Ivan do Carmo Machado,2023,NaN,ALICE: Uma infrAestrutura para meLhoria da qua...,Descrição: Este projeto propõe a construção de...,Em andamento,Pesquisa,Graduação: (2) / Doutorado: (3),Ivan do Carmo Machado - Coordenador / Larissa ...,Conselho Nacional de Desenvolvimento Científic...,Este projeto propõe a construção de uma infrae...


In [10]:
df["ano_conclusao"].unique()

array([  nan, 2024., 2022., 2014., 2013., 2012., 2025., 2020., 2018.,
       2004., 2021., 2016., 2015., 2010., 2008., 2006., 2019., 2002.,
       2017., 2023., 2026.])

In [11]:
df["situacao"].unique()

<StringArray>
['Em andamento', 'Concluído']
Length: 2, dtype: str

#### Contagem alunos

In [12]:
def extrair_contagem_alunos(texto):
    if not isinstance(texto, str):
        return pd.Series([0, 0, 0, 0])
    
    grad = re.search(r'Graduação:\s*\(([0-9]+)\)', texto)
    mest_ac = re.search(r'Mestrado acadêmico:\s*\(([0-9]+)\)', texto)
    mest_prof = re.search(r'Mestrado profissional:\s*\(([0-9]+)\)', texto)
    dout = re.search(r'Doutorado:\s*\(([0-9]+)\)', texto)
    
    n_grad = int(grad.group(1)) if grad else 0
    n_mest = (int(mest_ac.group(1)) if mest_ac else 0) + (int(mest_prof.group(1)) if mest_prof else 0)
    n_dout = int(dout.group(1)) if dout else 0
    n_total = n_grad + n_mest + n_dout
    
    return pd.Series([n_grad, n_mest, n_dout, n_total])

In [13]:
df[['qtd_graduacao', 'qtd_mestrado', 'qtd_doutorado', 'total_alunos']] = df['descricao'].apply(extrair_contagem_alunos)

In [14]:
df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa,qtd_graduacao,qtd_mestrado,qtd_doutorado,total_alunos
0,4430958315746203,Ivan do Carmo Machado,2024,NaN,Uma Investigação Empírica da Relação entre Ref...,Descrição: A temática de refatoração tem sido ...,Em andamento,Pesquisa,Mestrado acadêmico: (3) Doutorado: (4),Ivan do Carmo Machado - Coordenador / Carla Il...,Conselho Nacional de Desenvolvimento Científic...,A temática de refatoração tem sido amplamente ...,0,3,4,7
1,4430958315746203,Ivan do Carmo Machado,2024,NaN,Consolidação da Rede Internacional de Pesquisa...,Descrição: O objetivo do projeto é promover a ...,Em andamento,Pesquisa,Mestrado acadêmico: (26) Doutorado: (26),Ivan do Carmo Machado - Integrante / Climene L...,NaN,O objetivo do projeto é promover a formação ac...,0,26,26,52
2,4430958315746203,Ivan do Carmo Machado,2023,NaN,Melhoria da Qualidade de Software na Perspecti...,Descrição: O projeto vista investigar test sme...,Em andamento,Pesquisa,Graduação: (2) / Mestrado acadêmico: (2) / Dou...,Ivan do Carmo Machado - Integrante / rohit Ghe...,Conselho Nacional de Desenvolvimento Científic...,O projeto vista investigar test smells no cont...,2,2,6,10
3,4430958315746203,Ivan do Carmo Machado,2023,NaN,"SI2 - Instituto de Ciência, Tecnologia e Inova...",Descrição: Este projeto tem como objetivo gera...,Em andamento,Pesquisa,NaN,Ivan do Carmo Machado - Integrante / Eduardo S...,NaN,Este projeto tem como objetivo geral a promoçã...,0,0,0,0
4,4430958315746203,Ivan do Carmo Machado,2023,NaN,ALICE: Uma infrAestrutura para meLhoria da qua...,Descrição: Este projeto propõe a construção de...,Em andamento,Pesquisa,Graduação: (2) / Doutorado: (3),Ivan do Carmo Machado - Coordenador / Larissa ...,Conselho Nacional de Desenvolvimento Científic...,Este projeto propõe a construção de uma infrae...,2,0,3,5


In [15]:
txt = df.iloc[1,9]
len(txt.split("/"))

22

### LLM feature extraction

area_pgcomp: Subárea principal de pesquisa no PGCOMP.

edital_ou_chamada: Nome do edital/chamada identificado no texto.

tem_fomento: Booleano (Sim/Não) ou agência identificada no texto.

palavras_chave_llm: Lista de palavras-chave padronizadas.

parceria_internacional: Booleano (Sim/Não) e país/instituição parceira.

resumo_executivo: Síntese concisa do objetivo.

In [16]:
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU em uso:", torch.cuda.get_device_name(0))

CUDA disponível: True
GPU em uso: NVIDIA GeForce MX130


f:\Projetos\Mestrado\projects_analysis\.venv\Lib\site-packages\torch\cuda\__init__.py:484: UserWarning: Found GPU0 NVIDIA GeForce MX130 which is of compute capability (CC) 5.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.14.1+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
f:\Projetos\Mestrado\projects_analysis\.venv\Lib\site-packages\torch\cuda\__init__.py:602: UserWarning: 
NVIDIA GeForce MX

In [17]:
# huggingface token
load_dotenv()
# print("token: ", os.getenv("HF_TOKEN"))

True

In [18]:
# confirm if the token belogs to the user
# from huggingface_hub import HfApi

# user_info = HfApi().whoami(token=os.getenv("HF_TOKEN"))
# print(f"Token ativo para a conta: {user_info['name']}")

In [19]:
# model_id = "google/gemma-2-2b-it"
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu",
    local_files_only=True
)
model

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [20]:
## simple testing
input_text = "Write me a poem about Machine Learning."
input_ids = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(**input_ids, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Write me a poem about Machine Learning. Sure, I'd be happy to write you a poem about machine learning! Here's something for you:

In the realm of data and algorithms,
Where numbers dance with patterns so fine,
A new dawn breaks in the world of AI,
As machines learn


In [21]:
system = """
<role>
Você é um assistente acadêmico especialista em mineração de dados da Plataforma Lattes e análise de projetos de pesquisa do programa de mestrado de Ciência da Computação da UFBA.
</role>

<instrucoes>
1. Analise o texto do projeto fornecido e extraia ou deduza os campos definidos no schema JSON.
2. Idioma: Retorne todas as respostas em Português do Brasil.
3. Ausência de Informação: Se uma informação não estiver presente e não puder ser inferida com segurança, atribua o valor null ao campo (nunca invente dados).
4. Formato de Saída: Responda APENAS com o objeto JSON válido, sem texto explicativo antes ou depois, e sem marcações de código markdown (como ```json).
</instrucoes>

<taxonomia_pgcomp>
Para o campo 'area_pgcomp', escolha APENAS UMA das seguintes áreas do PGCOMP/UFBA que melhor descreve o projeto:
- Engenharia de Software
- Inteligência Artificial e Aprendizado de Máquina
- Redes de Computadores e Sistemas Distribuídos
- Ciência de Dados e Banco de Dados
- Segurança da Informação e Criptografia
- Visualização de Dados e Processamento de Imagens
- Computação Aplicada e Sistemas Interativos
- Otimização e Teoria da Computação
- Outra
</taxonomia_pgcomp>

<schema_json>
{
  "edital_ou_chamada": "string | null (Nome do edital ou chamada pública citada no texto, ex: 'Edital Universal CNPq 09/2023')",
  "coordenador": "string | null (Nome do coordenador do projeto. Extraia da lista de integrantes se houver a marcação 'Coordenador')",
  "financiador": "array | null (Lista com os nomes das agências de fomento, bolsas ou instituições financiadoras, ex: ['CNPq', 'CAPES'])",
  "area_pgcomp": "string (Apenas uma área contida na <taxonomia_pgcomp>)",
  "topico_pesquisa": "string (Tema/tópico específico do projeto, ex: 'Testes e Qualidade de Software')",
  "edital_ou_chamada": "string | null",
  "palavras_chave": ["array com 3 a 5 palavras-chave técnicas em Português"],
  "cooperacao_internacional": "boolean (true se houver colaboração com universidades/instituições estrangeiras, senão false)",
  "resumo_executivo": "string (Resumo conciso de 1 a 2 frases com o objetivo principal do projeto)"
}
</schema_json>

<exemplo>
Entrada do Usuário:
"Descrição: O projeto visa investigar test smells no contexto dos testes manuais e automatizados de software para propor técnicas e estratégias. (Chamada CNPq 14/2023 Apoio a Projetos Internacionais). Integrantes: Fulano de Tal - Coordenador / Beltrano - Integrante. Financiador(es): CNPq - Auxílio financeiro."

Saída Esperada:
{
  "edital_ou_chamada": "Chamada CNPq 14/2023 Apoio a Projetos Internacionais",
  "coordenador": "Fulano de Tal",
  "financiador": ["CNPq"],
  "area_pgcomp": "Engenharia de Software",
  "topico_pesquisa": "Teste de Software",
  "palavras_chave": ["Test Smells", "Testes de Software", "Qualidade de Software"],
  "cooperacao_internacional": true,
  "resumo_executivo": "O projeto investiga a ocorrência de test smells em testes manuais e automatizados de software para propor métodos e estratégias para diminuição do seu impacto na qualidade."
}
</exemplo>
"""

In [22]:
def processar_lote_llm(lote_textos: list[str]) -> list[dict]:
    prompts = []
    for texto in lote_textos:
        messages = [
            {"role": "user", "content": f"{system}\n\nTexto do Projeto:\n{texto}"}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)
    
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cpu")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    resultados = []
    input_len = inputs.input_ids.shape[1]
    for output in outputs:
        resposta_bruta = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
        print(f"respposta bruta:\n{resposta_bruta}")
        try:
            dados = json.loads(re.sub(r'^```json\s*|^```\s*|\s*```$', '', resposta_bruta, flags=re.MULTILINE).strip())
        except Exception:
            dados = {}
        resultados.append(dados)
        
    return resultados

In [28]:
from tqdm.auto import tqdm

In [ ]:
textos_limpos = df['descricao'].tolist()
BATCH_SIZE = 1
resultados_llm = []
arquivo_checkpoint = "checkpoint_resultados.jsonl"

print(f"Processando {len(textos_limpos)} projetos em lotes de {BATCH_SIZE}...")

loop_progress = tqdm(range(0, len(textos_limpos), BATCH_SIZE), desc="Processando Projetos")

for i in loop_progress:
    lote = textos_limpos[i:i + BATCH_SIZE]
    
    init = time.time()
    res_lote = processar_lote_llm(lote)
    resultados_llm.extend(res_lote)
    fim = time.time()
    
    tempo = fim - init
    
    # Atualiza a barra com o tempo do último lote, em vez de usar print()
    loop_progress.set_postfix(tempo_lote=f"{tempo:.1f}s")
    
    # Gravação incremental
    with open(arquivo_checkpoint, "a", encoding="utf-8") as f:
        for res in res_lote:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

# --- ETAPA 4: MERGE FINAL ---
df_llm = pd.DataFrame(resultados_llm)
df_final = pd.concat([df, df_llm], axis=1)

df_final.head()

Processando 171 projetos em lotes de 1...


Processando Projetos:   0%|          | 0/171 [00:00<?, ?it/s]

respposta bruta:
```json
{
  "edital_ou_chamada": "Situação: Em andamento; Natureza: Pesquisa",
  "coordenador": "Ivan do Carmo Machado",
  "financiador": ["Conselho Nacional de Desenvolvimento Científico e Tecnológico - Bolsa"],
  "area_pgcomp": "Visualização de Dados e Processamento de Imagens",
  "topico_pesquisa": "Refatoração de Testes",
  "palavras_chave": ["refatoração", "teste de software", "qualidade de teste"],
  "cooperacao_internacional": false,
  "resumo_executivo": "Este projeto investigará a refatoração de testes no contexto de sistemas de teste e suas implicações na qualidade e eficiência do código de teste."
}
```
respposta bruta:
```json
{
  "edital_ou_chamada": "Em andamento",
  "coordenador": "Climene Laura de Camargo",
  "financiador": ["Universidade Portuguesa"],
  "area_pgcomp": "Outra",
  "topico_pesquisa": "Formação Acadêmica de Negros e Indígenas",
  "palavras_chave": ["Formação Acadêmica", "Negros", "Indígenas", "Internationalization of Brazilian Universities

In [ ]:
print(df_final.shape)
print(df_final.columns.tolist())
save_path = os.path.join("data", "output")
if not os.path.exists(save_path):
    os.mkdir(save_path)
df_final.to_csv(os.path.join(save_path, 'projetos_lattes_completo.csv'), sep=';', index=False, encoding='utf-8-sig')
print("Processamento concluído com sucesso!")

(171, 24)
['lattes_id', 'professor', 'ano_inicio', 'ano_conclusao', 'projeto', 'descricao', 'situacao', 'natureza', 'alunos_envolvidos', 'integrantes', 'financiadores', 'descricao_limpa', 'qtd_graduacao', 'qtd_mestrado', 'qtd_doutorado', 'total_alunos', 'edital_ou_chamada', 'coordenador', 'financiador', 'area_pgcomp', 'topico_pesquisa', 'palavras_chave', 'cooperacao_internacional', 'resumo_executivo']
Processamento concluído com sucesso!
